In [21]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.model_selection import ParameterGrid

import sys
from itertools import product
sys.path.append('../../')
import src.forecasting.simulations      as sim
import src.fda.kde.estimators           as kde
import src.fda.transformations.lqdt     as lqdt
import src.fda.dfpc              as dfpc
import src.forecasting.pipelines        as fp
import src.forecasting.models           as fm

In [2]:
# Define scenarios
gas_params = {
    # "scenario_1": {
    #     "Description": "Location-driven ($m_t$)",
    #     "alpha": np.diag([0.08, 0.01, 0.01]),
    #     "beta": np.diag([0.90, 0.95, 0.95]),
    # },
    # "scenario_2": {
    #     "Description": "Scale-driven ($\sigma_t$)",
    #     "alpha": np.diag([0.01, 0.08, 0.01]),
    #     "beta": np.diag([0.95, 0.90, 0.95]),
    # },
    # "scenario_3": {
    #     "Description": "Shape-driven ($\eta_t$)",
    #      "alpha": np.diag([0.01, 0.01, 0.08]),
    #       "beta": np.diag([0.95, 0.95, 0.90])
    # },
    "scenario_4": {
        "Description": "Mixture",
        "alpha": np.diag([0.04, 0.06, 0.04]),
        "beta": np.diag([0.92, 0.95, 0.94]),
    }
}

distribution_params = {
    "nu": [3]
}

keys = ["scenario", "nu"]
values = [list(gas_params.keys()), distribution_params["nu"]]

param_grid = []

for scenario, nu in product(*values):
    gas_cfg = gas_params[scenario]

    param_grid.append({
        "scenario": "___nu=".join([scenario, str(nu)]),
        "nu": nu,
        "alpha": gas_cfg["alpha"],
        "beta": gas_cfg["beta"]    
})

In [3]:
# grid for densities
x = np.linspace(-40, 40, 5001)
# number of curves (densities)
T = 301
# number of simulations (f_{N_REP,1},...,f_{N_REP,T})
N_REPS = 10
# number of samples from each f_t density
N_SAMPLES = 288

total = len(param_grid) * N_REPS

In [4]:
sim_database = {}
total = len(param_grid) * N_REPS

for params in param_grid:
    scenario = params["scenario"]
    # Initialize scenario level
    sim_database[scenario] = {
        "params": params,
        "replications": {}
    }
    
    for n_rep in range(N_REPS):
        # 1. Setup Model and Simulate
        gm = sim.GasModel(alpha=params["alpha"], beta=params["beta"], nu=params["nu"])
        sim_results = gm.simulate(T=T, burn_in=300)
        
        # 2. Get Theoretical Densities
        sim_density = gm.conditional_densities(grid=x, theta_path=sim_results["theta"])
        dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=T, freq="D")
        sim_density.columns = dates
        
        # 3. Generate Samples Efficiently
        # Collect arrays first, then create DataFrame once
        samples_list = []
        for i in range(len(sim_results["theta"])):
            # Draw n samples for the theta at time i
            sample = gm.rvs(n=N_SAMPLES, theta=sim_results["theta"][i])
            samples_list.append(sample)
        
        # Create DataFrame: each column is a time step, each row a sample
        df_samples = pd.DataFrame(np.array(samples_list).T, columns=dates)
        
        # 4. Store in Database
        sim_database[scenario]["replications"][n_rep] = {
            "theta": sim_results["theta"],
            "densities": sim_density,
            "samples": df_samples
        }

In [5]:
returns_df = sim_database['scenario_4___nu=3']['replications'][0]['samples']
# bandwidths
kde_bw_params = {"method": "rot", "kernel": "gaussian", "sigma_robust":False}

df_h = kde.df_bandwidth_selector(returns_df, **kde_bw_params)    
kde_params = {k: v for k, v in kde_bw_params.items() if k in ['kernel', 'df']}

# kdes
df_grids, df_densities = kde.df_to_kde(
    X=returns_df, 
    h=df_h, 
    normalize_densities=False,
    **kde_params
)

In [6]:
mlqdt = lqdt.mLQDT()
model_lqd = mlqdt.transform(
    densities=df_densities,
    densities_supports=df_grids, 
    verbose=False
    )
# model_lqd.densities_to_lqdensities(verbose=False)

# 2. L2 Expansion (K_dFPC)
Y_t  = model_lqd.lqd.copy()
Y_t.index = model_lqd.lqd_support

/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: overflow encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


In [7]:
KdFPC_kwargs = {
    "p": 5,
    "dimension": 3
}

In [17]:
Y_train, Y_test = Y_t.iloc[:,:-1], Y_t.iloc[:,-1]
returns_train, returns_test = returns_df.iloc[:,:-1], returns_df.iloc[:,-1]

In [9]:
lqd_values  = Y_train.values
lqd_support = Y_train.index.values

KdFPC_kwargs.update({
    "u": lqd_support,
    "du": model_lqd.du
})

model_kdfpc = dfpc.K_dFPC(lqd_values)
model_kdfpc.fit(**KdFPC_kwargs)

In [19]:
k_etahat_fc = fp.run_multivariate_forecaster(model_kdfpc.etahat, 3, 'bic', 1, selected_nlags=3)
k_etahat_fc

array([[-0.12983518],
       [-0.17476064],
       [ 0.00454907]])

In [31]:
forecaster_en = fm.ScoreForecaster(model="ridge", lag=3)
forecaster_en.fit(returns_train.values, model_kdfpc.etahat.values)

forecaster_ada = fm.ScoreForecaster(model="adalasso", lag=3)
forecaster_ada.fit(returns_train.values, model_kdfpc.etahat.values)

pred_en = forecaster_en.predict_next(returns_train.values)
pred_ada = forecaster_ada.predict_next(returns_train.values)

print(pred_en)
print(pred_ada)

[[ 2.02811138]
 [ 0.13806456]
 [-0.05985487]]
[[ 1.30691708e+00]
 [ 1.01089402e-02]
 [-1.12575034e-03]]
